# 시연 3-0. 분류 모델로 성형 조건 추천하기

2회차에서 만든 불량 분류 모델을 **거꾸로** 씀: "이 조건이면 불량일까?" 대신 "불량이 안 나는 조건 범위는 어디일까?"

- 조건 하나만 바꿔 가며(나머지는 기준값) 모델이 예측하는 불량 확률을 구함
- 예측 불량 확률이 5% 아래인 구간을 **권장 조건 범위**로 삼음
- 결과를 저장하면 가상 공장의 `추천 조건 적용` 버튼이 이 파일을 읽음

In [ ]:
import sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
while not (ROOT / "common").exists():      # 저장소 최상위 폴더 찾기
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from common.plot_style import use_korean_font
use_korean_font()

DATA = ROOT / "data" / "session3"
OUT = DATA / "work"                        # 내 산출물이 저장되는 곳
OUT.mkdir(exist_ok=True)
pd.set_option("display.width", 160)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from common import process_model as pm
from common.recommend import CONDITIONS, recommend, sweep

S2 = ROOT / "data" / "session2"
src = S2 / "work" / "injection_clean.csv"
if not src.exists():
    src = S2 / "injection_clean_provided.csv"
df = pd.read_csv(src)
KO = {"mold_temp_c": "금형온도 (℃)", "melt_temp_c": "수지온도 (℃)", "inj_pressure_mpa": "사출압 (MPa)",
      "hold_pressure_mpa": "보압 (MPa)", "hold_time_s": "보압 시간 (s)", "cooling_time_s": "냉각 시간 (s)"}
model = RandomForestClassifier(n_estimators=200, min_samples_leaf=3, random_state=0, n_jobs=-1)
model.fit(df[CONDITIONS], df["is_defect"])
print(src.name, df.shape)

## 1. 조건을 하나씩 바꿔 보기

회색 띠가 예측 불량 확률 5% 아래인 구간(권장 범위), 점선이 현재 기준값임.

In [ ]:
LIMIT = 0.05                              # 허용하는 예측 불량 확률
base = dict(pm.INJ_NOMINAL)               # 나머지 조건을 고정해 둘 값
reco = recommend(model, df, base, LIMIT)
fig, axes = plt.subplots(2, 3, figsize=(14, 6.5))
for ax, key in zip(axes.ravel(), CONDITIONS):
    grid, p = sweep(model, df, base, key)
    ax.plot(grid, p * 100, color="#c0392b", lw=2)
    ax.axvspan(reco[key]["low"], reco[key]["high"], color="#2e86c1", alpha=0.15)
    ax.axvline(base[key], color="gray", ls="--", lw=1)
    ax.axhline(LIMIT * 100, color="gray", lw=0.8)
    ax.set_title(KO[key]); ax.set_ylabel("예측 불량 확률 (%)"); ax.grid(alpha=0.3)
plt.tight_layout(); fig.savefig(OUT / "recommend.png", dpi=120); plt.show()

## 2. 권장 조건 범위

In [ ]:
table = pd.DataFrame(reco).T.rename(columns={"low": "하한", "high": "상한", "recommend": "추천값"})
table.insert(0, "기준값", [base[k] for k in table.index])
table.index = [KO[k] for k in table.index]
(OUT / "recommend.json").write_text(__import__("json").dumps(reco, ensure_ascii=False, indent=1))
table

## 3. 가상 공장에서 확인

1. 가상 공장 패널의 `현장 변동 (금형온도 · 보압 저하)` → 불량이 늘어나는 것을 확인
2. `추천 조건 적용` → 위 표의 추천값이 슬라이더에 들어감 → 불량이 줄어드는 것을 확인

> 모델은 **데이터가 있었던 범위 안에서만** 믿을 수 있음. 그래서 범위를 데이터의 1~99% 구간으로 제한했음.
> 추천은 "권고"임. 설비에 바로 쓰기 전에 공정 담당자가 확인하는 단계가 필요함 (4회차의 제어 3수준).